# RetailS shoplifting — pose-sequence anomaly detection

One-class setup: train only on *normal* pose sequences (`RetailS_train`), score anomalies on the two
labelled test splits (`RetailS_test_staged`, `RetailS_test_realworld`).

**Set `MODE` in section 0** to pick the model - `"poseflow"` (graph normalizing flow, scores by
negative log-likelihood) or `"lstm_ae"` (LSTM autoencoder, scores by reconstruction MSE). Training,
checkpoint name and scoring all follow from it; nothing else needs editing to switch. Section 6
explains the difference and why it matters here.

One training sample = one fixed-length window of pose (`SEG_LEN` consecutive frames) belonging to a
**single track_id** (one person). Windows never mix people.

Sections: 0 setup · 1 raw format · 2 EDA · 3 preprocessing · 4 dataset · 5 eval infra · 6 models ·
7 train · 8 evaluate.

## 0. Setup

Data lives in Drive only as a zip (`RetailS.zip` under
`/content/drive/MyDrive/Liquir_Store/dev/person-shoplifting/`); it gets extracted once to
`/content/RetailS` (local Colab disk) so every training epoch reads from fast local storage instead
of paying Drive I/O latency per file. Only checkpoints/models write back to Drive, under a `models/`
folder in that same project directory (created automatically if missing).

In [ ]:
import os
import glob

# ─── pick the model to train/evaluate ────────────────────────────────────────────
#   "poseflow"  - graph normalizing flow, scores by negative log-likelihood (density modeling)
#   "lstm_ae"   - LSTM autoencoder, scores by reconstruction MSE (the original baseline)
# Everything downstream (checkpoint name, training loss, anomaly score) follows from this.
MODE = "poseflow"
assert MODE in ("poseflow", "lstm_ae")
# ─────────────────────────────────────────────────────────────────────────────────

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

PROJECT_DIR = "/content/drive/MyDrive/Liquir_Store/dev/person-shoplifting"

def find_data_root(search_root):
    """The zip may or may not contain an extra top-level 'RetailS/' folder -
    walk the extracted tree and use whichever directory directly contains
    RetailS_train, rather than assuming a fixed nesting depth."""
    for dirpath, dirnames, _ in os.walk(search_root):
        if "RetailS_train" in dirnames:
            return dirpath
    raise FileNotFoundError(f"no RetailS_train/ found anywhere under {search_root}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_PATH = "/content/drive/MyDrive/Liquir_Store/data/RetailS.zip"   # <-- adjust filename if different
    LOCAL_EXTRACT_ROOT = "/content/RetailS"       # extracted locally: fast I/O, no per-file Drive latency

    if not os.path.isdir(LOCAL_EXTRACT_ROOT):
        import zipfile
        print(f"extracting {ZIP_PATH} -> {LOCAL_EXTRACT_ROOT} ...")
        with zipfile.ZipFile(ZIP_PATH) as zf:
            zf.extractall(LOCAL_EXTRACT_ROOT)
    else:
        print(f"{LOCAL_EXTRACT_ROOT} already extracted, skipping unzip")

    DATA_ROOT = find_data_root(LOCAL_EXTRACT_ROOT)

    # checkpoints/models persist to Drive - the only I/O that should cross the Drive boundary
    MODELS_DIR = f"{PROJECT_DIR}/models"
    os.makedirs(MODELS_DIR, exist_ok=True)
else:
    DATA_ROOT = os.path.join(os.path.dirname(os.getcwd()), "datasets", "RetailS")
    MODELS_DIR = os.path.join(os.path.dirname(os.getcwd()), "outputs", "retails_train")
    os.makedirs(MODELS_DIR, exist_ok=True)

TRAIN_DIR = f"{DATA_ROOT}/RetailS_train/pose/train"
TEST_STAGED_DIR = f"{DATA_ROOT}/RetailS_test_staged/pose/test"
TEST_STAGED_GT = f"{DATA_ROOT}/RetailS_test_staged/gt/test_frame_mask"
TEST_RW_DIR = f"{DATA_ROOT}/RetailS_test_realworld/pose/test"
TEST_RW_GT = f"{DATA_ROOT}/RetailS_test_realworld/gt/test_frame_mask"

CKPT_PATH = f"{MODELS_DIR}/{MODE}_best.pt"   # per-mode, so switching MODE never clobbers the other

print("MODE:      ", MODE)
print("DATA_ROOT (local, fast):", DATA_ROOT)
print("checkpoint (Drive, persisted):", CKPT_PATH)
assert os.path.isdir(TRAIN_DIR), f"train dir not found: {TRAIN_DIR} - fix DATA_ROOT/ZIP_PATH above"

In [ ]:
# !pip install -q torch scikit-learn matplotlib tqdm   # usually already present on Colab

import json
import glob
import random

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available()
                       else "mps" if torch.backends.mps.is_available()
                       else "cpu")
print("device:", DEVICE)

## 1. Inspect the raw format

Each train JSON: `{track_id: {frame_idx: {"keypoints": [...51 floats...], "scores": null}}}` -
17 keypoints in COCO joint order (nose, eyes, ears, shoulders, elbows, wrists, hips, knees, ankles),
the same joint order YOLO-pose (`models/yolo*-pose.onnx`) outputs, so no joint remapping is needed.

**But each triplet is `(y, x, conf)`, not `(x, y, conf)`** - row-major image indexing. This isn't
documented in the RetailS README; it was measured (see `track_to_array` in section 3, and
`scripts/retails_visualize.py`). Read the wrong way round, every skeleton comes out transposed -
people lying sideways - and the height-based normalization below silently becomes width-based.

In [ ]:
COCO_KP_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow",
    "left_wrist", "right_wrist", "left_hip", "right_hip",
    "left_knee", "right_knee", "left_ankle", "right_ankle",
]
COCO_SKELETON = [
    (15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12),
    (5, 6), (5, 7), (6, 8), (7, 9), (8, 10), (1, 2), (0, 1), (0, 2),
    (1, 3), (2, 4), (3, 5), (4, 6),
]

sample_f = sorted(glob.glob(f"{TRAIN_DIR}/*.json"))[0]
d = json.load(open(sample_f))
tid0 = list(d.keys())[0]
frame0 = sorted(d[tid0].keys(), key=int)[0]
kp = d[tid0][frame0]["keypoints"]
print("file:", sample_f)
print("tracks in file:", len(d), "| first track id:", tid0, "| frames in it:", len(d[tid0]))
print("keypoints length:", len(kp), "-> num keypoints:", len(kp) / 3)
assert len(kp) == 17 * 3, "unexpected keypoint layout - re-check before trusting COCO order below"

In [ ]:
def draw_skeleton(ax, kp51, conf_thr=0.2, title=""):
    # (y, x, conf) in the file -> (x, y) for plotting; see track_to_array in section 3 for why.
    kp = np.array(kp51, dtype=np.float32).reshape(17, 3)
    xy, conf = kp[:, [1, 0]], kp[:, 2]
    for a, b in COCO_SKELETON:
        if conf[a] > conf_thr and conf[b] > conf_thr:
            ax.plot([xy[a, 0], xy[b, 0]], [xy[a, 1], xy[b, 1]], "-", color="tab:blue")
    ax.scatter(xy[:, 0], xy[:, 1], c=conf, cmap="RdYlGn", vmin=0, vmax=1, s=25, zorder=3)
    ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_title(title)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
frames_sorted = sorted(d[tid0].keys(), key=int)
for ax, fr in zip(axes, frames_sorted[::max(1, len(frames_sorted) // 4)][:4]):
    draw_skeleton(ax, d[tid0][fr]["keypoints"], title=f"track {tid0} frame {fr}")
plt.tight_layout(); plt.show()

## 2. EDA — split sizes, camera coverage, track length, label balance

In [ ]:
def count_files(d):
    return len(glob.glob(f"{d}/*.json"))

print(f"train:          {count_files(TRAIN_DIR)} files (normal only, no labels)")
print(f"test_staged:    {count_files(TEST_STAGED_DIR)} files (labelled)")
print(f"test_realworld: {count_files(TEST_RW_DIR)} files (labelled)")

In [ ]:
def contiguous_runs(frame_dict):
    """A track's frame indices split into maximal runs with no gaps."""
    idxs = sorted(int(k) for k in frame_dict.keys())
    runs, run = [], [idxs[0]]
    for a, b in zip(idxs, idxs[1:]):
        if b - a == 1:
            run.append(b)
        else:
            runs.append(run)
            run = [b]
    runs.append(run)
    return runs

n_tracks, run_lens = [], []
for f in tqdm(sorted(glob.glob(f"{TRAIN_DIR}/*.json"))[:200], desc="sampling 200 train files"):
    d = json.load(open(f))
    n_tracks.append(len(d))
    for frames in d.values():
        run_lens.extend(len(r) for r in contiguous_runs(frames))
run_lens = np.array(run_lens)

print(f"tracks/file: mean={np.mean(n_tracks):.1f}")
print(f"contiguous run length (frames): mean={run_lens.mean():.1f} median={np.median(run_lens):.0f} "
      f"min={run_lens.min()} max={run_lens.max()}")

_seg_len_preview = 12   # actual SEG_LEN is set in section 3 - this is just to size the histogram
print(f"runs usable at seg_len={_seg_len_preview}: {(run_lens >= _seg_len_preview).mean()*100:.1f}%")

plt.hist(run_lens, bins=50); plt.xlabel("contiguous run length (frames)"); plt.ylabel("count")
plt.axvline(_seg_len_preview, color="red", ls="--", label=f"seg_len={_seg_len_preview}")
plt.legend(); plt.show()

In [ ]:
def label_balance(pose_dir, gt_dir, label):
    files = sorted(glob.glob(f"{pose_dir}/*.json"))
    n_pos_clip, total_frames, pos_frames = 0, 0, 0
    for f in files:
        stem = os.path.splitext(os.path.basename(f))[0]
        gt = np.load(f"{gt_dir}/{stem}.npy")
        total_frames += len(gt); pos_frames += int(gt.sum())
        n_pos_clip += int(gt.sum() > 0)
    print(f"{label}: {n_pos_clip}/{len(files)} clips have >=1 anomalous frame "
          f"({100*n_pos_clip/len(files):.0f}%) | frame-level positive rate "
          f"{100*pos_frames/total_frames:.1f}%")

label_balance(TEST_STAGED_DIR, TEST_STAGED_GT, "test_staged")
label_balance(TEST_RW_DIR, TEST_RW_GT, "test_realworld")

## 3. Preprocessing (v2 — STG-NF-style)

The first pass (per-frame hip-anchored normalization, seg_len=24/stride=12, confidence as a loss
weight) came out at AUC-ROC ~0.47-0.50 - chance level, and reconstruction error was actually *lower*
on labelled shoplifting frames than normal ones. Comparing against the two purpose-built pose-anomaly
baselines the RetailS/Shopformer authors benchmark against - [GEPC](https://github.com/amirmk89/gepc)
and [STG-NF](https://github.com/orhir/STG-NF) (ICCV 2023, ~1K parameters, still SOTA on this exact
task) - the normalization is the main thing done differently here, not model capacity:

- **Center/scale per window, not per frame.** STG-NF centers on the mean of *every* keypoint across
  the whole window and scales by the **std of the Y-coordinate over the whole window** - not a
  per-frame hip anchor. This is far less sensitive to camera distance/angle drift than re-anchoring
  every single frame to its own hip midpoint.
- **Shorter, denser windows.** `seg_len=12, stride=1` (both GEPC and STG-NF default to this) instead
  of `24/12` - catches localized hand/arm motion instead of averaging it out over a long block.
- **Drop low-confidence windows, don't weight them.** Both reference repos filter out segments whose
  mean keypoint confidence falls below a threshold, rather than down-weighting them in the loss.

Train windows still only come from `RetailS_train`; test clips are still scored whole via a sliding
window in section 7, each candidate window re-normalized independently the same way.

In [ ]:
SEG_LEN = 12     # was 24 - shorter window, GEPC/STG-NF default
SEG_STRIDE = 3   # train-set stride; was 1 (~6M windows, ~10GB+ raw, OOM on 12.7GB RAM). stride=1 is
                 # only needed at eval time (scoring every frame of a clip) - training windows this
                 # close together barely add diversity, so a coarser stride here is a memory fix, not
                 # a quality cut. ~2M windows / ~3.4GB raw at stride=3, leaves headroom on 12.7GB RAM.
                 # Eval's sliding_infer always steps 1 frame at a time regardless of this constant.
CONF_TH = 0.3    # window mean-confidence below this is dropped, not weighted (p10 of per-frame mean conf)

def track_to_array(frame_dict, idxs):
    """-> (T, 17, 3) as [x, y, conf].

    RetailS stores each keypoint triplet as (y, x, conf) - row-major image indexing, not (x, y).
    Verified over 196k high-confidence frames: read as (y, x), head sits above shoulders above hips
    above ankles in 94.5% of them with a median height/width of 2.75 (a standing person); read as
    (x, y) that ordering holds only 47% of the time - chance - at a ratio of 0.36, i.e. everyone
    lying sideways. Getting this wrong silently swaps the body's vertical and horizontal axes, which
    matters below: normalize_window scales by the std of Y on purpose (body height is the stable
    reference), and scaling by width instead is both smaller and far noisier - it collapses whenever
    someone turns to face the camera.
    """
    kp = np.array([frame_dict[str(i)]["keypoints"] for i in idxs],
                  dtype=np.float32).reshape(-1, 17, 3)
    return np.stack([kp[..., 1], kp[..., 0], kp[..., 2]], axis=-1)

def normalize_window(kps):
    """kps: (T, 17, 3) as [x, y, conf] -> (feat (T,34), mean_confidence)

    STG-NF-style: center on the mean of every keypoint across the whole window, scale by the
    std of the Y-coordinate across the whole window. Computed per-window (not per-frame, not
    per-track), so it captures relative motion inside that window and is far less sensitive to
    where in the frame / how far from the camera the window happens to sit.
    """
    xy, conf = kps[..., :2], kps[..., 2]
    center = xy.reshape(-1, 2).mean(axis=0)
    scale = xy[..., 1].std()
    scale = scale if scale >= 1e-3 else 1.0
    xy_norm = (xy - center) / scale
    return xy_norm.reshape(len(kps), -1).astype(np.float32), float(conf.mean())

def build_train_windows(json_dir, seg_len=SEG_LEN, stride=SEG_STRIDE, conf_th=CONF_TH, max_files=None):
    files = sorted(glob.glob(f"{json_dir}/*.json"))
    if max_files:
        files = files[:max_files]
    windows, n_dropped = [], 0
    for f in tqdm(files, desc="building train windows"):
        d = json.load(open(f))
        for frames in d.values():
            for run in contiguous_runs(frames):
                if len(run) < seg_len:
                    continue
                arr = track_to_array(frames, run)          # (L, 17, 3) raw, [x, y, conf]
                for start in range(0, len(arr) - seg_len + 1, stride):
                    feat, mean_conf = normalize_window(arr[start:start + seg_len])
                    if mean_conf < conf_th:
                        n_dropped += 1
                        continue
                    windows.append(feat)
    print(f"dropped {n_dropped} low-confidence windows (mean conf < {conf_th})")
    return windows

In [ ]:
# Rough size estimate before committing to a full build - catches an OOM-prone SEG_STRIDE/SEG_LEN
# choice before it crashes the kernel, instead of after. Uses the 200-file run_lens sample from
# the EDA cell above, scaled up to the full train file count.
_n_train_files = len(glob.glob(f"{TRAIN_DIR}/*.json"))
_est_windows = np.maximum(0, (run_lens - SEG_LEN) // SEG_STRIDE + 1).sum() * (_n_train_files / 200)
_est_gb = _est_windows * SEG_LEN * 34 * 4 / 1e9   # float32, raw data only (list/object overhead adds more)
print(f"estimated windows on all {_n_train_files} train files: ~{_est_windows:,.0f}  "
      f"(~{_est_gb:.1f} GB raw float32 - actual RAM usage will run higher due to Python list overhead)")

# Smoke-test on a small subset first - flip to None for the full run once this looks right.
MAX_TRAIN_FILES = 50

windows = build_train_windows(TRAIN_DIR, max_files=MAX_TRAIN_FILES)
print(f"windows built: {len(windows)} from {MAX_TRAIN_FILES or 'all'} files")
print("window shape:", windows[0].shape)

## 4. Dataset / DataLoader

In [ ]:
class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        return torch.from_numpy(self.windows[i])

random.shuffle(windows)
n_val = max(1, int(0.1 * len(windows)))
val_windows, train_windows = windows[:n_val], windows[n_val:]

train_loader = DataLoader(WindowDataset(train_windows), batch_size=128, shuffle=True)
val_loader = DataLoader(WindowDataset(val_windows), batch_size=256, shuffle=False)
print(f"train windows: {len(train_windows)} | val windows: {len(val_windows)}")

## 5. Eval infrastructure — shared by both models

`frame_idx` inside a test clip's JSON is an **absolute position in the original clip**, not a local
index into that track's own sequence - a track that only appears from frame 20-50 of an 80-frame clip
still uses keys "20".."50". Some clips also have more than one track (staged: 163/624, realworld:
20/47) - a bystander or a second person in frame alongside the labelled one.

`eval_split` takes a `score_fn(frames_dict) -> {absolute_frame_idx: anomaly_score}` for a single
track, so it works with whichever model `MODE` selects. Each track is scored on its own contiguous
runs, then per-frame scores are scattered into a `len(gt)`-long array at their real `frame_idx`.
Frames covered by more than one track take the max score across tracks (a frame is anomalous if
anyone in it is). Frames no track covers are left out of the metric - there's nothing to score there.

Frame-level scores + ground truth are pooled across all clips in a split before computing metrics,
the same way the reference papers report AUC-ROC / AUC-PR / EER.

In [ ]:
def eval_split(pose_dir, gt_dir, label, score_fn, max_files=None):
    """score_fn(frames_dict) -> {absolute_frame_idx: anomaly_score}, one call per track."""
    files = sorted(glob.glob(f"{pose_dir}/*.json"))
    if max_files:
        files = files[:max_files]
    y_true, y_score, n_multitrack, n_dropped = [], [], 0, 0
    for f in tqdm(files, desc=f"scoring {label}"):
        stem = os.path.splitext(os.path.basename(f))[0]
        gt = np.load(f"{gt_dir}/{stem}.npy")
        L = len(gt)
        d = json.load(open(f))
        n_multitrack += len(d) > 1

        full = np.full(L, np.nan)
        for frames in d.values():
            for fidx, s in score_fn(frames).items():
                if 0 <= fidx < L:
                    full[fidx] = s if np.isnan(full[fidx]) else max(full[fidx], s)
                else:
                    n_dropped += 1               # track frame outside the labelled clip range

        mask = ~np.isnan(full)
        y_true.append(gt[mask]); y_score.append(full[mask])
    y_true, y_score = np.concatenate(y_true), np.concatenate(y_score)

    auc_roc = roc_auc_score(y_true, y_score)
    auc_pr = average_precision_score(y_true, y_score)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer = (fpr[idx] + fnr[idx]) / 2

    print(f"{label}: {n_multitrack}/{len(files)} clips had >1 track "
          f"({n_dropped} track-frames fell outside the clip and were dropped)")
    print(f"{label}: AUC-ROC={auc_roc:.3f}  AUC-PR={auc_pr:.3f}  EER={eer:.3f}  "
          f"(n_frames={len(y_true)}, positive_rate={y_true.mean()*100:.1f}%)")
    return dict(y_true=y_true, y_score=y_score, fpr=fpr, tpr=tpr,
                auc_roc=auc_roc, auc_pr=auc_pr, eer=eer)

## 6. Models — whichever `MODE` selected in section 0

Both are defined here; `MODE` at the top of the notebook decides which one gets built, trained and
scored. Nothing else needs changing to switch.

**`lstm_ae`** — LSTM autoencoder, anomaly score = reconstruction MSE. The original baseline. It
scored AUC-ROC ~0.45-0.51 on both test splits across two different normalization schemes - chance
level, with reconstruction error actually *lower* on labelled shoplifting frames than on normal ones.
That's the classic failure mode of reconstruction-based anomaly detection: an autoencoder
reconstructs an anomaly just as well as normal data, especially when the anomaly is kinematically
*simpler* (standing still to conceal something vs. varied walking). Kept as a comparison point.

**`poseflow`** — a graph normalizing flow, anomaly score = **negative log-likelihood**. GEPC and
STG-NF both score *how likely a pose is under a learned distribution* rather than how well it
reconstructs, which sidesteps that failure mode. This is a **simplified, single-scale** version of
STG-NF's idea, not a port of their multi-scale Glow stack (that scaffolding is aimed at image-sized
inputs and isn't needed to test density-vs-reconstruction on a window this small). What is faithful:

- **Graph**: coupling layers condition through a small ST-GCN block (graph conv over the 17-joint
  COCO skeleton + temporal conv over the window), not a plain MLP.
- **Flow**: exact log-likelihood via change-of-variables (ActNorm + affine coupling, invertible both
  ways), trained by maximizing `log p(x)` on normal windows only.

Note both runs before this point were trained on **axis-swapped** poses (the `(y, x)` bug fixed in
section 1/3), so their numbers should be re-measured rather than trusted.

In [ ]:
import math

# ── lstm_ae ──────────────────────────────────────────────────────────────────────
class LSTMAutoencoder(nn.Module):
    """Encode the window to one latent vector, decode back to the full sequence."""
    def __init__(self, input_dim=34, hidden=64, latent=32, num_layers=1):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden, num_layers, batch_first=True)
        self.to_latent = nn.Linear(hidden, latent)
        self.from_latent = nn.Linear(latent, hidden)
        self.decoder = nn.LSTM(hidden, hidden, num_layers, batch_first=True)
        self.out = nn.Linear(hidden, input_dim)

    def forward(self, x):               # x: (B, T, 34) -> reconstruction (B, T, 34)
        T = x.size(1)
        _, (h, _) = self.encoder(x)
        z = self.to_latent(h[-1])
        dec_in = self.from_latent(z).unsqueeze(1).repeat(1, T, 1)
        dec_out, _ = self.decoder(dec_in)
        return self.out(dec_out)


# ── poseflow ─────────────────────────────────────────────────────────────────────
def build_adjacency(skeleton_edges, num_nodes=17):
    """Symmetric-normalized adjacency (self-loops + COCO skeleton bones), the standard ST-GCN graph."""
    A = np.eye(num_nodes, dtype=np.float32)
    for a, b in skeleton_edges:
        A[a, b] = 1.0
        A[b, a] = 1.0
    deg = A.sum(axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(deg))
    return (D_inv_sqrt @ A @ D_inv_sqrt).astype(np.float32)

A_NORM = torch.from_numpy(build_adjacency(COCO_SKELETON)).to(DEVICE)   # (17, 17)


class STGCNBlock(nn.Module):
    """One spatio-temporal graph conv block: per-joint linear -> mix across the joint graph ->
    temporal conv across the window. Input/output: (B, C, T, V)."""
    def __init__(self, in_ch, out_ch, A, t_kernel=3):
        super().__init__()
        self.A = A
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1)
        self.tcn = nn.Conv2d(out_ch, out_ch, kernel_size=(t_kernel, 1), padding=(t_kernel // 2, 0))
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.proj(x)
        x = torch.einsum("bctv,vw->bctw", x, self.A)
        x = self.act(x)
        return self.tcn(x)


class ActNorm2d(nn.Module):
    """Per-channel affine, invertible, with a tracked log-determinant. No data-dependent init here
    (kept simple) - starts as identity and learns via gradient descent like the rest of the model."""
    def __init__(self, num_channels):
        super().__init__()
        self.log_scale = nn.Parameter(torch.zeros(1, num_channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, num_channels, 1, 1))

    def forward(self, x):
        y = (x + self.bias) * torch.exp(self.log_scale)
        T, V = x.shape[2], x.shape[3]
        logdet = (self.log_scale.sum() * T * V).expand(x.size(0))
        return y, logdet


class AffineCouplingSTGCN(nn.Module):
    """Splits the 2 coordinate channels (x, y): one stays fixed and conditions, through an
    ST-GCN block, the scale/shift applied to the other. `swap` alternates which channel is fixed
    across flow steps, standing in for the permutation layer real Glow-style flows use."""
    def __init__(self, A, hidden=16, swap=False):
        super().__init__()
        self.swap = swap
        self.net = nn.Sequential(STGCNBlock(1, hidden, A), STGCNBlock(hidden, 2, A))

    def forward(self, x):   # x: (B, 2, T, V)
        xa, xb = (x[:, 0:1], x[:, 1:2]) if not self.swap else (x[:, 1:2], x[:, 0:1])
        params = self.net(xa)
        log_s, t = torch.tanh(params[:, 0:1]), params[:, 1:2]   # tanh clamp: keeps exp(log_s) stable
        yb = xb * torch.exp(log_s) + t
        y = torch.cat([xa, yb], dim=1) if not self.swap else torch.cat([yb, xa], dim=1)
        logdet = log_s.sum(dim=[1, 2, 3])
        return y, logdet


class PoseFlow(nn.Module):
    """K steps of (ActNorm -> ST-GCN affine coupling). forward() returns exact log p(x) per
    window under a standard-normal base distribution, via the change-of-variables formula."""
    def __init__(self, A, K=8, hidden=16):
        super().__init__()
        self.actnorms = nn.ModuleList([ActNorm2d(2) for _ in range(K)])
        self.couplings = nn.ModuleList([AffineCouplingSTGCN(A, hidden, swap=(i % 2 == 1)) for i in range(K)])

    def forward(self, x):   # x: (B, 2, T, V) -> log p(x), shape (B,)
        logdet_total = torch.zeros(x.size(0), device=x.device)
        z = x
        for actnorm, coupling in zip(self.actnorms, self.couplings):
            z, ld1 = actnorm(z)
            z, ld2 = coupling(z)
            logdet_total = logdet_total + ld1 + ld2
        log_pz = (-0.5 * z ** 2 - 0.5 * math.log(2 * math.pi)).sum(dim=[1, 2, 3])
        return log_pz + logdet_total


def window_to_flow_input(feat_batch):
    """(B, T, 34) normalized window features -> (B, 2, T, 17) for the flow model.
    34 = 17 keypoints * (x, y), interleaved in COCO order (matches normalize_window's output)."""
    B, T = feat_batch.shape[0], feat_batch.shape[1]
    return feat_batch.view(B, T, 17, 2).permute(0, 3, 1, 2).contiguous()


# ── build whichever MODE asked for, plus its loss and its per-frame scoring rule ──
model = (PoseFlow(A_NORM, K=8, hidden=16) if MODE == "poseflow" else LSTMAutoencoder()).to(DEVICE)

def batch_loss(feat):
    """feat: (B, T, 34) on DEVICE -> scalar loss to minimize."""
    if MODE == "poseflow":
        return -model(window_to_flow_input(feat)).mean()
    return nn.functional.mse_loss(model(feat), feat)

def window_score(feat):
    """feat: (T, 34) one normalized window (numpy) -> per-frame anomaly score (T,).

    The AE gives a genuine per-frame error; the flow gives one likelihood for the window as a
    whole, so that single value is broadcast across the window's frames. Overlapping windows are
    averaged per frame by sliding_infer either way, so the two stay directly comparable.
    """
    x = torch.from_numpy(feat[None]).float().to(DEVICE)
    with torch.no_grad():
        if MODE == "poseflow":
            nll = float(-model(window_to_flow_input(x)).item())
            return np.full(len(feat), nll)
        rec = model(x)[0].cpu().numpy()
        return ((rec - feat) ** 2).mean(-1)

print(f"MODE={MODE}  ->  {type(model).__name__}, "
      f"{sum(p.numel() for p in model.parameters()):,} parameters")

## 7. Train

Normal-only: minimize reconstruction MSE (`lstm_ae`) or negative log-likelihood (`poseflow`), both
through the same `batch_loss`. Best-by-validation checkpoint goes to `CKPT_PATH`, which is named per
`MODE` so switching modes never overwrites the other model's weights.

In [ ]:
EPOCHS = 20 if MODE == "poseflow" else 30
LR = 1e-3
LOSS_NAME = "NLL (-log p(x))" if MODE == "poseflow" else "MSE"

opt = torch.optim.Adam(model.parameters(), lr=LR)
train_hist, val_hist, best_val = [], [], float("inf")

for epoch in range(EPOCHS):
    model.train()
    tr, n = 0.0, 0
    for feat in train_loader:
        feat = feat.to(DEVICE)
        loss = batch_loss(feat)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)   # flows can spike early on
        opt.step()
        tr += loss.item() * feat.size(0); n += feat.size(0)
    tr /= n

    model.eval()
    va, n = 0.0, 0
    with torch.no_grad():
        for feat in val_loader:
            feat = feat.to(DEVICE)
            va += batch_loss(feat).item() * feat.size(0); n += feat.size(0)
    va /= n

    train_hist.append(tr); val_hist.append(va)
    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), CKPT_PATH)     # written straight to Drive
    print(f"epoch {epoch+1:2d}/{EPOCHS}  train={tr:.4f}  val={va:.4f}"
          f"{'  (best)' if va == best_val else ''}")

plt.plot(train_hist, label="train"); plt.plot(val_hist, label="val")
plt.xlabel("epoch"); plt.ylabel(LOSS_NAME); plt.title(f"{MODE} training"); plt.legend(); plt.show()
print("best checkpoint:", CKPT_PATH)

## 8. Evaluate

`sliding_infer` walks each contiguous run one frame at a time, re-normalizing every candidate window
independently (exactly as training windows were built - normalizing a whole track upfront would leak
whole-track statistics into each window), scores it via `window_score` (MSE or NLL depending on
`MODE`), and averages overlapping windows per frame. That feeds `eval_split` from section 5.

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()

def sliding_infer(raw_kps, seg_len=SEG_LEN):
    """raw_kps: (T, 17, 3) raw [x, y, conf], one contiguous run -> per-frame score (T,)."""
    T = raw_kps.shape[0]
    if T <= seg_len:
        arr_p = np.pad(raw_kps, ((0, seg_len - T), (0, 0), (0, 0)), mode="edge")
        feat, _ = normalize_window(arr_p)
        return window_score(feat)[:T]
    acc, count = np.zeros(T), np.zeros(T)
    for start in range(0, T - seg_len + 1):
        feat, _ = normalize_window(raw_kps[start:start + seg_len])
        acc[start:start + seg_len] += window_score(feat)
        count[start:start + seg_len] += 1
    return acc / np.maximum(count, 1)

def score_track(frames_dict, seg_len=SEG_LEN):
    """One track -> {absolute_frame_idx: anomaly_score}, scored per contiguous run."""
    out = {}
    for run in contiguous_runs(frames_dict):
        out.update(zip(run, sliding_infer(track_to_array(frames_dict, run), seg_len)))
    return out

res_staged = eval_split(TEST_STAGED_DIR, TEST_STAGED_GT, "test_staged", score_fn=score_track)
res_rw = eval_split(TEST_RW_DIR, TEST_RW_GT, "test_realworld", score_fn=score_track)

In [ ]:
plt.plot(res_staged["fpr"], res_staged["tpr"], label=f"test_staged (AUC={res_staged['auc_roc']:.3f})")
plt.plot(res_rw["fpr"], res_rw["tpr"], label=f"test_realworld (AUC={res_rw['auc_roc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.legend(); plt.title(f"ROC — {MODE}"); plt.show()

# Which way the score actually points. If the anomalous class scores LOWER on average, the model is
# not just weak - it is anti-correlated, which is what the reconstruction AE did on this dataset.
for name, r in (("test_staged", res_staged), ("test_realworld", res_rw)):
    pos = r["y_score"][r["y_true"] == 1]
    neg = r["y_score"][r["y_true"] == 0]
    print(f"{name:15s} mean score  shoplifting={pos.mean():10.4f}  normal={neg.mean():10.4f}  "
          f"-> {'correct direction' if pos.mean() > neg.mean() else 'INVERTED'}")

## Next steps

1. Both earlier runs (AUC-ROC ~0.45-0.51) used **axis-swapped** poses - the `(y, x)` bug fixed in
   sections 1/3 meant the height-based normalization was silently dividing by body *width*. Re-run
   both modes on the corrected data before drawing any conclusion about either model.
2. Set `MAX_TRAIN_FILES = None` in section 3 for the full train set once a smoke run looks right.
3. Watch the score-direction check printed under the ROC. An AUC near 0.5 with `INVERTED` direction
   means the score is anti-correlated with the label, not merely uninformative - a different problem
   from a weak-but-correct signal, and the reconstruction AE's known failure mode here.
4. If `poseflow` on corrected data still sits near 0.5 in the correct direction, that points at the
   input rather than the model: raw body pose may not separate this dataset's shoplifting frames, and
   the next lever is richer input (hand-to-object proximity, held-item detection, optical flow).
5. `K` (flow steps) and `hidden` (ST-GCN width) are the first knobs if training NLL plateaus high -
   both are cheap given the model is only a few thousand parameters.